# Детекция ботов по событиям Авито

Для каждой куки из `test.csv` нужно выдать `score`. Чем он выше, тем больше кука похожа на автоматический сбор объявлений. Я проверял качество по Precision при Recall не ниже 70%, той же функцией, что дали в задании (`metric.py`). На платформе финальный файл получил 0,81360. Меток скрытого теста у меня нет.

Сначала я попробовал простую базовую модель по двум счётчикам. Потом добавил время и действия. Больше всего помогли признаки общих объявлений и похожих маршрутов разных куки. В финале смешал CatBoost (60%) и HistGradientBoosting (40%). Всего получилось 132 признака. Ниже код, который заново создаёт отправленный `submission.csv`. Отдельно проверяю границы суток, дубли и порядок `cookie_id`.

Запускать ячейки нужно сверху вниз из папки проекта. Рядом должны лежать `data/`, `metric.py` и `sample_submission.csv`. Проверял на Python 3.13.5. Версии библиотек записал в `requirements.txt`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score
from threadpoolctl import threadpool_limits

from metric import precision_at_recall

SEED = 0
DATA_DIR = Path('data')
assert (DATA_DIR / 'train.csv').exists(), 'Запустите ноутбук из папки проекта'
print('Python:', sys.version.split()[0])

Python: 3.13.5


## 1. Сначала смотрю на данные

В `train.csv` и `test.csv` одна строка соответствует одной куке и её суточному окну. События лежат отдельно и могут выходить за это окно. Дни теста идут после обучающих, поэтому проверяю модель на более поздних днях.

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
events = pd.read_csv(DATA_DIR / 'events.csv.gz')
sample = pd.read_csv('sample_submission.csv')

for frame in (train, test):
    for col in ('cookie_created_at', 'window_start_ts', 'window_end_ts'):
        frame[col] = pd.to_datetime(frame[col], errors='raise')
    assert frame.cookie_id.is_unique
events['event_ts'] = pd.to_datetime(events.event_ts, errors='raise')
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert set(train.target.unique()) == {0, 1}

display(pd.DataFrame({
    'строк': [len(train), len(test), len(events)],
    'первый день': [train.window_start_ts.min(), test.window_start_ts.min(), events.event_ts.min()],
    'последний день': [train.window_start_ts.max(), test.window_start_ts.max(), events.event_ts.max()],
}, index=['train', 'test', 'events']))
print('Ботов в train:', int(train.target.sum()), 'из', len(train))
print('Порядок sample и test совпадает:', sample.cookie_id.equals(test.cookie_id))

,строк,первый день,последний день
train,11091,2026-04-06 00:00:00,2026-04-19 00:00:00
test,4909,2026-04-20 00:00:00,2026-04-26 00:00:00
events,328905,2026-04-06 00:02:57,2026-04-26 23:59:50


Ботов в train: 899 из 11091
Порядок sample и test совпадает: False


## 2. Отбираю события ровно внутри окна

По условию беру только события с `window_start_ts <= event_ts < window_end_ts`. Если просто собрать всю историю куки, в признаки попадут лишние события. Точные повторы строк тоже есть. Готовлю два варианта событий. С повторами считаю исходные счётчики и интервалы времени, как в финальной модели. Без повторов считаю остальные признаки. Разные написания платформы привожу к нижнему регистру, но исходное поле не теряю.

In [3]:
EVENT_COLUMNS = tuple(events.columns)
WINDOW_COLUMNS = ['cookie_id', 'window_start_ts', 'window_end_ts']


def prepare_events(raw_events, metadata, *, drop_exact_duplicates=False):
    """Отобрать события окна, опционально убрать точные повторы, упорядочить."""
    windows = metadata[WINDOW_COLUMNS].copy()  # target сюда не попадает
    assert windows.cookie_id.is_unique
    assert windows.notna().all().all()
    assert windows.window_start_ts.lt(windows.window_end_ts).all()
    assert raw_events[['cookie_id', 'event_ts']].notna().all().all()
    scoped = raw_events.loc[raw_events.cookie_id.isin(windows.cookie_id), list(EVENT_COLUMNS)].copy()
    ev = scoped.merge(windows, on='cookie_id', how='left', validate='many_to_one', sort=False)
    in_window = ev.event_ts.ge(ev.window_start_ts) & ev.event_ts.lt(ev.window_end_ts)
    ev = ev.loc[in_window].copy()
    n_inside = len(ev)
    duplicate_mask = ev.duplicated(subset=list(EVENT_COLUMNS), keep='first')
    n_duplicates = int(duplicate_mask.sum())
    if drop_exact_duplicates:
        ev = ev.loc[~duplicate_mask].copy()

    normalized = ev.platform.astype('string').str.strip().str.casefold()
    ev['platform_normalized'] = normalized.mask(normalized.eq(''), pd.NA)
    ev = ev.sort_values(['cookie_id', 'event_ts'], kind='stable').reset_index(drop=True)

    assert ev.event_ts.ge(ev.window_start_ts).all()
    assert ev.event_ts.lt(ev.window_end_ts).all()
    assert not ev.groupby('cookie_id', sort=False).event_ts.diff().lt(pd.Timedelta(0)).any()
    if drop_exact_duplicates:
        assert not ev.duplicated(subset=list(EVENT_COLUMNS)).any()
    audit = {
        'input_rows_for_cookies': len(scoped),
        'outside_window': len(scoped) - n_inside,
        'rows_in_window': n_inside,
        'exact_duplicates': n_duplicates,
        'duplicates_removed': n_duplicates if drop_exact_duplicates else 0,
        'output_rows': len(ev),
        'cookies_without_events': len(set(windows.cookie_id) - set(ev.cookie_id)),
    }
    return ev, audit

In [4]:
train_events, train_audit = prepare_events(events, train)
train_unique, train_unique_audit = prepare_events(events, train, drop_exact_duplicates=True)
test_events, test_audit = prepare_events(events, test)
test_unique, test_unique_audit = prepare_events(events, test, drop_exact_duplicates=True)
display(pd.DataFrame({
    'train': train_audit, 'train без дублей': train_unique_audit,
    'test': test_audit, 'test без дублей': test_unique_audit,
}).T)
print('После нормализации:', sorted(train_unique.platform_normalized.dropna().unique()))

,input_rows_for_cookies,outside_window,rows_in_window,exact_duplicates,duplicates_removed,output_rows,cookies_without_events
train,239215,40779,198436,2952,0,198436,0
train без дублей,239215,40779,198436,2952,2952,195484,0
test,89690,0,89690,1341,0,89690,0
test без дублей,89690,0,89690,1341,1341,88349,0


После нормализации: ['android', 'desktop', 'ios', 'iphone', 'web']


Пропуски не заполняю одним общим нулём. Например, у события в мобильном приложении может не быть координат курсора. Это нормально. Если действия не было, его счётчик равен нулю. Если паузу посчитать нельзя, оставляю `NaN` в признаках. CatBoost обрабатывает пропуски сам, а перед HistGradientBoosting я заменяю их на -1.

In [5]:
# Пропуски полезнее смотреть с учётом типа события и платформы.
missing_by_context = (train_unique.assign(
    cursor_present=train_unique.pointer_x.notna() & train_unique.pointer_y.notna())
    .groupby(['event_name', 'platform_normalized'], dropna=False)
    .agg(events=('cookie_id', 'size'), cursor_present=('cursor_present', 'mean'),
         item_present=('item_id', lambda x: x.notna().mean()),
         query_present=('search_query', lambda x: x.notna().mean()))
    .sort_values('events', ascending=False).head(12))
display(missing_by_context.round(3))

events  ...  query_present
event_name          platform_normalized          ...               
item_view           web                   30939  ...            0.0
                    android               28032  ...            0.0
search_results_view web                   25761  ...            1.0
                    android               23182  ...            1.0
item_view           desktop               10440  ...            0.0
photo_swipe         android                9366  ...            0.0
                    web                    8931  ...            0.0
search_results_view desktop                8734  ...            1.0
favorite_add        android                4888  ...            0.0
                    web                    4682  ...            0.0
seller_page_view    web                    4357  ...            0.0
                    android                4180  ...            0.0

[12 rows x 4 columns]

## 3. Простая базовая модель

Для точки отсчёта беру только число событий и число разных объявлений. Обучаю случайный лес на днях до 17 апреля и проверяю на 17–19 апреля. Это похоже на пример в `quickstart.ipynb` и помогает оценить пользу следующих признаков.

In [6]:
def basic_features(ev, metadata):
    grouped = ev.groupby('cookie_id')
    features = pd.DataFrame({
        'n_events': grouped.size(),
        'item_nunique': grouped.item_id.nunique(),
    })
    return (metadata[['cookie_id']]
            .merge(features.reset_index(), on='cookie_id', how='left', validate='one_to_one')
            .fillna({'n_events': 0, 'item_nunique': 0}))

In [7]:
baseline_features = basic_features(train_events, train)
late_start = pd.Timestamp('2026-04-17')
late_end = pd.Timestamp('2026-04-20')
fit_mask = train.window_end_ts.le(late_start)
valid_mask = train.window_start_ts.ge(late_start) & train.window_start_ts.lt(late_end)
assert train.loc[fit_mask, 'window_end_ts'].max() <= train.loc[valid_mask, 'window_start_ts'].min()

forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED)
forest.fit(baseline_features.loc[fit_mask, ['n_events', 'item_nunique']],
           train.loc[fit_mask, 'target'])
baseline_score = forest.predict_proba(
    baseline_features.loc[valid_mask, ['n_events', 'item_nunique']])[:, 1]
print('Baseline, 17–19 апреля:', round(precision_at_recall(
    train.loc[valid_mask, 'target'], baseline_score), 6))

Baseline, 17–19 апреля: 0.10247


С двумя простыми счётчиками на позднем периоде получилось 0,10247. Количество событий само по себе почти не помогает отличить ботов от людей. Дальше добавляю признаки поведения, сохраняя ту же временную схему проверки.

## 4. Признаки самой куки

Дальше смотрю на поведение куки. Считаю паузы между событиями, действия, разнообразие объявлений и запросов, а также короткие переходы вроде «поиск -> объявление». Из User-Agent беру только категорию браузера и ОС. Саму строку в модель не передаю. Курсор считаю только там, где координаты вообще имеют смысл. Каждая функция возвращает одну строку на куку и не использует `target`.

В функциях также учитываю куки, у которых не было подходящих событий.

In [8]:
TIME_COLUMNS = [
    'activity_span_s', 'gap_mean_s', 'gap_median_s', 'gap_p10_s', 'gap_p90_s',
    'gap_max_s', 'gap_std_s', 'gap_cv', 'zero_gap_fraction', 'short_gap_fraction',
    'active_minutes', 'active_hours', 'peak_minute_fraction', 'session_count',
]
SHORT_GAP_SECONDS = 5
SESSION_BREAK_SECONDS = 30 * 60


def temporal_features(ev, metadata):
    """Временные признаки для каждой куки. События уже отобраны по окнам."""
    ordered = ev[['cookie_id', 'event_ts']].copy()
    ordered = ordered.sort_values(['cookie_id', 'event_ts'], kind='stable')
    groups = ordered.groupby('cookie_id', sort=False)
    first_last = groups.event_ts.agg(first='min', last='max')
    features = pd.DataFrame(index=first_last.index)
    features['activity_span_s'] = (first_last['last'] - first_last['first']).dt.total_seconds()

    ordered['gap_s'] = groups.event_ts.diff().dt.total_seconds()
    gaps = ordered.loc[ordered.gap_s.notna(), ['cookie_id', 'gap_s']]
    gap_groups = gaps.groupby('cookie_id').gap_s
    stats = gap_groups.agg(gap_mean_s='mean', gap_median_s='median', gap_max_s='max')
    stats['gap_p10_s'] = gap_groups.quantile(0.10)
    stats['gap_p90_s'] = gap_groups.quantile(0.90)
    stats['gap_std_s'] = gap_groups.std(ddof=0)
    stats['gap_cv'] = stats.gap_std_s.div(stats.gap_mean_s.where(stats.gap_mean_s.gt(0)))
    stats['zero_gap_fraction'] = gaps.gap_s.eq(0).groupby(gaps.cookie_id).mean()
    stats['short_gap_fraction'] = gaps.gap_s.le(SHORT_GAP_SECONDS).groupby(gaps.cookie_id).mean()
    features = features.join(stats)

    ordered['minute'] = ordered.event_ts.dt.floor('min')
    ordered['hour'] = ordered.event_ts.dt.floor('h')
    features['active_minutes'] = ordered.groupby('cookie_id').minute.nunique()
    features['active_hours'] = ordered.groupby('cookie_id').hour.nunique()
    minute_counts = ordered.groupby(['cookie_id', 'minute']).size()
    features['peak_minute_fraction'] = minute_counts.groupby(level='cookie_id').max().div(groups.size())
    long_pauses = gaps.gap_s.gt(SESSION_BREAK_SECONDS).groupby(gaps.cookie_id).sum()
    features['session_count'] = 1 + long_pauses.reindex(features.index, fill_value=0)

    result = metadata[['cookie_id']].merge(features.reset_index(), on='cookie_id',
                                          how='left', validate='one_to_one')
    counts = ['active_minutes', 'active_hours', 'session_count']
    result[counts] = result[counts].fillna(0).astype('int64')
    result = result[['cookie_id', *TIME_COLUMNS]]
    return result

In [9]:
ACTION_TYPES = [
    'search_results_view', 'item_view', 'photo_swipe', 'seller_page_view',
    'contact_phone_show', 'contact_chat_open', 'contact_message_sent',
    'favorite_add', 'login',
]
ACTION_BUCKETS = [*ACTION_TYPES, 'other']
ACTION_COUNT_COLUMNS = [f'event_count__{name}' for name in ACTION_BUCKETS]


def action_features(ev, metadata):
    """Сколько раз кука сделала каждое действие внутри своего окна."""
    bucket = ev.event_name.where(ev.event_name.isin(ACTION_TYPES), 'other')
    counts = pd.crosstab(ev.cookie_id, bucket).reindex(
        index=pd.Index(metadata.cookie_id, name='cookie_id'),
        columns=ACTION_BUCKETS, fill_value=0).astype('int64')
    counts.columns.name = None
    result = counts.add_prefix('event_count__').reset_index()
    return result

In [10]:
VIEW_CONTENT_COLUMNS = [
    'view_item_nunique', 'view_item_repeat_fraction', 'view_item_top_fraction', 'view_item_missing_fraction',
    'view_category_nunique', 'view_category_top_fraction', 'view_category_entropy',
    'view_location_nunique', 'view_location_top_fraction', 'view_location_entropy',
]
SEARCH_CONTENT_COLUMNS = [
    'search_query_nunique', 'search_query_unique_fraction', 'search_query_top_fraction',
    'search_page_max', 'search_page_median', 'query_page_nunique',
    'query_page_repeat_fraction', 'search_forward_step_fraction',
]
CONTENT_COLUMNS = [*VIEW_CONTENT_COLUMNS, *SEARCH_CONTENT_COLUMNS]


def value_diversity(frame, column):
    """Распределение непустых значений одного поля внутри куки."""
    counts = frame.dropna(subset=[column]).groupby(['cookie_id', column]).size()
    total = counts.groupby(level='cookie_id').sum()
    unique = counts.groupby(level='cookie_id').size()
    probabilities = counts.div(total, level='cookie_id')
    entropy = -(probabilities * np.log(probabilities)).groupby(level='cookie_id').sum()
    normalized_entropy = entropy.div(np.log(unique.where(unique.gt(1))))
    normalized_entropy = normalized_entropy.where(unique.ne(1), 0).clip(0, 1)
    return pd.DataFrame({
        'observations': total, 'nunique': unique,
        'top_fraction': counts.groupby(level='cookie_id').max().div(total),
        'entropy': normalized_entropy,
    })


def content_features(ev, metadata):
    """Признаки объявлений и запросов. События уже отобраны по окнам и очищены от повторов."""
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name='cookie_id'))
    views = ev.loc[ev.event_name.eq('item_view')].copy()
    item_stats = value_diversity(views, 'item_id')
    result['view_item_nunique'] = item_stats['nunique']
    result['view_item_repeat_fraction'] = 1 - item_stats['nunique'].div(item_stats.observations)
    result['view_item_top_fraction'] = item_stats.top_fraction
    result['view_item_missing_fraction'] = views.item_id.isna().groupby(views.cookie_id).mean()
    for column, prefix in [('item_category', 'view_category'), ('item_location', 'view_location')]:
        stats = value_diversity(views, column)
        for statistic in ['nunique', 'top_fraction', 'entropy']:
            result[f'{prefix}_{statistic}'] = stats[statistic]

    search = ev.loc[ev.event_name.eq('search_results_view')].copy()
    query = search.search_query.astype('string').str.casefold().str.strip().str.replace(r'\s+', ' ', regex=True)
    search['query_normalized'] = query.mask(query.eq(''), pd.NA)
    query_stats = value_diversity(search, 'query_normalized')
    result['search_query_nunique'] = query_stats['nunique']
    result['search_query_unique_fraction'] = query_stats['nunique'].div(query_stats.observations)
    result['search_query_top_fraction'] = query_stats.top_fraction
    pages = pd.to_numeric(search.search_page, errors='raise')
    search['valid_page'] = pages.where(pages.ge(1) & pages.eq(np.floor(pages)))
    result['search_page_max'] = search.groupby('cookie_id').valid_page.max()
    result['search_page_median'] = search.groupby('cookie_id').valid_page.median()
    pairs = search.dropna(subset=['query_normalized', 'valid_page'])
    pair_counts = pairs.groupby(['cookie_id', 'query_normalized', 'valid_page']).size()
    unique_pairs = pair_counts.groupby(level='cookie_id').size()
    result['query_page_nunique'] = unique_pairs
    result['query_page_repeat_fraction'] = 1 - unique_pairs.div(pair_counts.groupby(level='cookie_id').sum())

    # Один момент времени может иметь несколько несовместимых значений страницы.
    # Помечаем весь момент, а не выбираем произвольную строку после сортировки.
    moments = pairs.groupby(['cookie_id', 'query_normalized', 'event_ts']).valid_page.agg(
        page='min', n_pages='nunique').reset_index()
    moments = moments.sort_values(['cookie_id', 'query_normalized', 'event_ts'], kind='stable')
    moments['page'] = moments.page.where(moments.n_pages.eq(1))
    previous = moments.groupby(['cookie_id', 'query_normalized']).page.shift()
    comparable = moments.page.notna() & previous.notna()
    forward = (moments.page - previous).eq(1)
    result['search_forward_step_fraction'] = forward.loc[comparable].groupby(
        moments.loc[comparable, 'cookie_id']).mean()

    count_columns = ['view_item_nunique', 'view_category_nunique', 'view_location_nunique',
                     'search_query_nunique', 'query_page_nunique']
    result[count_columns] = result[count_columns].fillna(0).astype('int64')
    result = result.reset_index()[['cookie_id', *CONTENT_COLUMNS]]
    return result

Разнообразие карточек и запросов считаю только по соответствующим событиям. Для повторяющихся значений беру долю самого частого и нормированную энтропию. Если у куки нет поисков, поисковые доли остаются пропущенными, а не превращаются в фиктивные нули.

In [11]:
BROWSER_FAMILIES = ['avito_app', 'edge', 'opera', 'yandex', 'chrome', 'firefox',
                    'safari', 'http_client', 'other', 'missing']
OS_FAMILIES = ['android', 'ios', 'windows', 'macos', 'linux', 'other', 'missing']
PLATFORM_FAMILIES = ['web', 'desktop', 'android', 'ios', 'iphone', 'other', 'missing']
CLIENT_COLUMNS = [
    *[f'ua_browser_{v}_fraction' for v in BROWSER_FAMILIES],
    *[f'ua_os_{v}_fraction' for v in OS_FAMILIES],
    *[f'platform_{v}_fraction' for v in PLATFORM_FAMILIES],
    'ua_browser_nunique', 'ua_os_nunique', 'platform_nunique', 'ua_headless_fraction',
]


def parse_user_agents(values):
    text = values.astype('string').str.strip().str.casefold()
    missing = text.isna() | text.eq('')
    s = text.fillna('')
    def contains(pattern):
        return s.str.contains(pattern, regex=True, na=False).to_numpy(dtype=bool)
    browser = np.select([
        missing.to_numpy(dtype=bool), contains(r'^avito/'),
        contains(r'edg[a-z]*/|edge/'), contains(r'opr/|opera/'), contains(r'yabrowser/'),
        contains(r'headlesschrome/|chrome/|crios/'), contains(r'firefox/|fxios/'),
        contains(r'safari/'), contains(r'python-requests/|python-urllib|aiohttp/|curl/|wget/|okhttp/'),
    ], ['missing', 'avito_app', 'edge', 'opera', 'yandex', 'chrome', 'firefox', 'safari', 'http_client'],
        default='other')
    os_family = np.select([
        missing.to_numpy(dtype=bool), contains(r'android'), contains(r'iphone|ipad|ipod'),
        contains(r'windows'), contains(r'macintosh|mac os x'), contains(r'linux'),
    ], ['missing', 'android', 'ios', 'windows', 'macos', 'linux'], default='other')
    return pd.DataFrame({'browser': browser, 'os': os_family,
        'headless': contains(r'headlesschrome/')}, index=values.index)


def client_features(ev, metadata):
    parsed = parse_user_agents(ev.user_agent)
    client = ev[['cookie_id']].copy()
    client['browser'] = parsed.browser
    client['os'] = parsed.os
    platform = ev.platform_normalized.astype('string')
    client['platform'] = platform.where(platform.isin(PLATFORM_FAMILIES[:-2]), 'other').mask(platform.isna(), 'missing')
    client['headless'] = parsed.headless
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name='cookie_id'))
    for field, categories, prefix in [
        ('browser', BROWSER_FAMILIES, 'ua_browser'), ('os', OS_FAMILIES, 'ua_os'),
        ('platform', PLATFORM_FAMILIES, 'platform')]:
        counts = pd.crosstab(client.cookie_id, client[field]).reindex(columns=categories, fill_value=0)
        fractions = counts.div(counts.sum(axis=1), axis=0)
        for category in categories:
            result[f'{prefix}_{category}_fraction'] = fractions[category]
        known = client.loc[client[field].ne('missing')]
        result[f'{prefix}_nunique'] = known.groupby('cookie_id')[field].nunique().reindex(result.index, fill_value=0)
    result['ua_headless_fraction'] = client.groupby('cookie_id').headless.mean()
    result = result.reset_index()[['cookie_id', *CLIENT_COLUMNS]]
    return result

In [12]:
SEQUENCE_ACTION_MAP = {
    'search_results_view': 'search', 'item_view': 'item', 'photo_swipe': 'photo',
    'seller_page_view': 'seller', 'contact_phone_show': 'contact',
    'contact_chat_open': 'contact', 'contact_message_sent': 'contact',
    'favorite_add': 'favorite', 'login': 'login',
}
SEQUENCE_COLUMNS = [
    'seq_moment_count', 'seq_pair_count', 'seq_triplet_count',
    'seq_ambiguous_moment_fraction', 'seq_pair_coverage_fraction',
    'seq_same_action_fraction', 'seq_transition_repeat_fraction', 'seq_transition_entropy',
    'seq_search_to_item_fraction', 'seq_item_to_contact_fraction', 'seq_item_to_photo_fraction',
    'seq_item_to_favorite_fraction', 'seq_contact_to_item_fraction', 'seq_search_item_contact_fraction',
]
MIN_SEQUENCE_OBSERVATIONS = 3


def sequence_features(ev, metadata):
    states = ev[['cookie_id', 'event_ts']].copy()
    states['action'] = ev.event_name.map(SEQUENCE_ACTION_MAP).fillna('other')
    moments = states.groupby(['cookie_id', 'event_ts']).action.agg(
        state='min', n_states='nunique').reset_index().sort_values(['cookie_id', 'event_ts'], kind='stable')
    moments['state'] = moments.state.where(moments.n_states.eq(1))
    grouped = moments.groupby('cookie_id')
    gap = grouped.event_ts.diff().dt.total_seconds()
    step_ok = gap.gt(0) & gap.le(SESSION_BREAK_SECONDS)
    previous = grouped.state.shift()
    previous2 = grouped.state.shift(2)
    moments['step_ok'] = step_ok
    previous_step_ok = moments.groupby('cookie_id').step_ok.shift().fillna(False).astype(bool)
    pair_ok = step_ok & previous.notna() & moments.state.notna()
    triple_ok = pair_ok & previous_step_ok & previous2.notna()
    pairs = moments.loc[pair_ok, ['cookie_id', 'state']].assign(source=previous.loc[pair_ok])
    triples = moments.loc[triple_ok, ['cookie_id', 'state']].assign(
        first=previous2.loc[triple_ok], second=previous.loc[triple_ok])
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name='cookie_id'))
    moment_count = grouped.size()
    pair_count = pairs.groupby('cookie_id').size()
    triple_count = triples.groupby('cookie_id').size()
    result['seq_moment_count'] = moment_count.reindex(result.index, fill_value=0)
    result['seq_pair_count'] = pair_count.reindex(result.index, fill_value=0)
    result['seq_triplet_count'] = triple_count.reindex(result.index, fill_value=0)
    result['seq_ambiguous_moment_fraction'] = moments.n_states.gt(1).groupby(moments.cookie_id).mean()
    possible_pairs = step_ok.groupby(moments.cookie_id).sum()
    result['seq_pair_coverage_fraction'] = pair_count.reindex(possible_pairs.index, fill_value=0).div(
        possible_pairs.where(possible_pairs.gt(0)))
    enough_pairs = pair_count.ge(MIN_SEQUENCE_OBSERVATIONS)
    result['seq_same_action_fraction'] = pairs.state.eq(pairs.source).groupby(pairs.cookie_id).mean().where(enough_pairs)
    transitions = pairs.groupby(['cookie_id', 'source', 'state']).size()
    unique = transitions.groupby(level='cookie_id').size()
    result['seq_transition_repeat_fraction'] = (1 - unique.div(pair_count)).where(enough_pairs)
    probabilities = transitions.div(pair_count, level='cookie_id')
    entropy = -(probabilities * np.log(probabilities)).groupby(level='cookie_id').sum()
    normalized = entropy.div(np.log(unique.where(unique.gt(1)))).where(unique.ne(1), 0)
    result['seq_transition_entropy'] = normalized.where(enough_pairs).clip(0, 1)
    for source, destination in [('search', 'item'), ('item', 'contact'), ('item', 'photo'),
                                ('item', 'favorite'), ('contact', 'item')]:
        source_pairs = pairs.loc[pairs.source.eq(source)]
        denominator = source_pairs.groupby('cookie_id').size()
        fraction = source_pairs.state.eq(destination).groupby(source_pairs.cookie_id).mean()
        result[f'seq_{source}_to_{destination}_fraction'] = fraction.where(
            denominator.ge(MIN_SEQUENCE_OBSERVATIONS))
    search_item = triples.loc[triples['first'].eq('search') & triples.second.eq('item')]
    denominator = search_item.groupby('cookie_id').size()
    result['seq_search_item_contact_fraction'] = search_item.state.eq('contact').groupby(
        search_item.cookie_id).mean().where(denominator.ge(MIN_SEQUENCE_OBSERVATIONS))
    result = result.reset_index()[['cookie_id', *SEQUENCE_COLUMNS]]
    return result

Для коротких цепочек сначала объединяю события с одинаковым временем. Если в одну секунду оказались разные действия, порядок между ними неизвестен. Такой переход пропускаю. Пауза больше 30 минут тоже разрывает цепочку.

In [13]:
CURSOR_COLUMNS = [
    'cursor_web_count', 'cursor_observed_count', 'cursor_observed_fraction',
    'cursor_item_observed_fraction', 'cursor_search_observed_fraction',
    'cursor_contact_observed_fraction', 'cursor_unique_fraction',
    'cursor_top_fraction', 'cursor_x_unique_fraction',
    'cursor_y_unique_fraction', 'cursor_x_std', 'cursor_y_std',
    'cursor_x_iqr', 'cursor_y_iqr', 'cursor_origin_fraction',
    'cursor_step_count', 'cursor_step_median', 'cursor_step_p90',
    'cursor_stationary_fraction',
]
FLOW_COLUMNS = [
    'seller_pro_fraction', 'seller_missing_fraction', 'seller_type_nunique',
    'contact_item_missing_fraction', 'contact_without_prior_view_fraction',
    'contact_item_repeat_fraction', 'contact_after_first_view_median_s',
    'item_action_diversity_mean',
]


CONTACT_EVENTS = ['contact_phone_show', 'contact_chat_open', 'contact_message_sent']


def add_cursor_features(ev, result):
    """Добавить признаки курсора в таблицу куки."""
    # Для курсора подходят только события веб-версии и desktop.
    web = ev.loc[ev.platform_normalized.isin(['web', 'desktop'])].copy()
    web['observed'] = web.pointer_x.notna() & web.pointer_y.notna()
    points = web.loc[web.observed].copy()
    web_count = web.groupby('cookie_id').size()
    point_count = points.groupby('cookie_id').size()
    result['cursor_web_count'] = web_count.reindex(result.index, fill_value=0)
    result['cursor_observed_count'] = point_count.reindex(result.index, fill_value=0)
    result['cursor_observed_fraction'] = (
        point_count.reindex(web_count.index, fill_value=0).div(web_count)
    )

    cursor_event_groups = [
        ('item', web.event_name.eq('item_view')),
        ('search', web.event_name.eq('search_results_view')),
        ('contact', web.event_name.isin(CONTACT_EVENTS)),
    ]
    for name, mask in cursor_event_groups:
        part = web.loc[mask]
        result[f'cursor_{name}_observed_fraction'] = (
            part.groupby('cookie_id')['observed'].mean()
        )

    positions = points.groupby(['cookie_id', 'pointer_x', 'pointer_y']).size()
    result['cursor_unique_fraction'] = (
        positions.groupby(level='cookie_id').size().div(point_count)
    )
    result['cursor_top_fraction'] = (
        positions.groupby(level='cookie_id').max().div(point_count)
    )
    for coordinate, axis in [('pointer_x', 'x'), ('pointer_y', 'y')]:
        grouped = points.groupby('cookie_id')[coordinate]
        result[f'cursor_{axis}_unique_fraction'] = grouped.nunique().div(point_count)
        result[f'cursor_{axis}_std'] = grouped.std(ddof=0).where(point_count.ge(3))
        result[f'cursor_{axis}_iqr'] = (
            grouped.quantile(.75) - grouped.quantile(.25)
        ).where(point_count.ge(3))
    result['cursor_origin_fraction'] = (
        (points.pointer_x.eq(0) & points.pointer_y.eq(0))
        .groupby(points.cookie_id).mean()
    )

    # Если в одну секунду записаны разные точки, движение между ними не угадываю.
    moments = (points.groupby(['cookie_id', 'event_ts'])
               .agg(x=('pointer_x', 'min'), y=('pointer_y', 'min'),
                    nx=('pointer_x', 'nunique'), ny=('pointer_y', 'nunique'))
               .reset_index()
               .sort_values(['cookie_id', 'event_ts']))
    coherent = moments.nx.eq(1) & moments.ny.eq(1)
    moments['x'] = moments.x.where(coherent)
    moments['y'] = moments.y.where(coherent)
    grouped = moments.groupby('cookie_id')
    time_gap = grouped.event_ts.diff().dt.total_seconds()
    dx = grouped.x.diff()
    dy = grouped.y.diff()
    distance = np.hypot(dx, dy).where(
        time_gap.gt(0) & time_gap.le(SESSION_BREAK_SECONDS)
    )
    valid = distance.notna()
    steps = distance.loc[valid].groupby(moments.loc[valid, 'cookie_id'])
    step_count = steps.size()
    result['cursor_step_count'] = step_count.reindex(result.index, fill_value=0)
    result['cursor_step_median'] = steps.median().where(step_count.ge(3))
    result['cursor_step_p90'] = steps.quantile(.9).where(step_count.ge(3))
    result['cursor_stationary_fraction'] = (
        distance.loc[valid].eq(0)
        .groupby(moments.loc[valid, 'cookie_id']).mean()
        .where(step_count.ge(3))
    )

    return result


def add_contact_features(ev, result):
    """Добавить признаки продавцов и контактов."""
    # Отдельно считаю просмотры продавцов и действия с контактами.
    views = ev.loc[ev.event_name.eq('item_view')].copy()
    sellers = views.seller_type.astype('string').str.strip().str.casefold()
    result['seller_missing_fraction'] = sellers.isna().groupby(views.cookie_id).mean()
    known_sellers = sellers.notna()
    result['seller_pro_fraction'] = (
        sellers.loc[known_sellers].eq('pro')
        .groupby(views.loc[known_sellers, 'cookie_id']).mean()
    )
    result['seller_type_nunique'] = (
        sellers.groupby(views.cookie_id).nunique().reindex(result.index, fill_value=0)
    )

    contacts = ev.loc[ev.event_name.isin(CONTACT_EVENTS)].copy()
    result['contact_item_missing_fraction'] = (
        contacts.item_id.isna().groupby(contacts.cookie_id).mean()
    )
    contacts = contacts.dropna(subset=['item_id'])
    contact_count = contacts.groupby('cookie_id').size()
    result['contact_item_repeat_fraction'] = (
        1 - contacts.groupby('cookie_id').item_id.nunique().div(contact_count)
    )
    first_view = (views.dropna(subset=['item_id'])
                  .groupby(['cookie_id', 'item_id']).event_ts.min()
                  .rename('first_view'))
    contacts = contacts.merge(first_view, on=['cookie_id', 'item_id'],
                              how='left', validate='many_to_one')
    had_prior_view = contacts.first_view.notna() & contacts.first_view.le(contacts.event_ts)
    result['contact_without_prior_view_fraction'] = (
        (~had_prior_view).groupby(contacts.cookie_id).mean()
    )
    delays = (contacts.event_ts - contacts.first_view).dt.total_seconds().where(had_prior_view)
    result['contact_after_first_view_median_s'] = delays.groupby(contacts.cookie_id).median()
    action_diversity = (ev.dropna(subset=['item_id'])
                        .groupby(['cookie_id', 'item_id']).event_name.nunique())
    result['item_action_diversity_mean'] = action_diversity.groupby(level='cookie_id').mean()

    return result


def richer_behaviour_features(ev, metadata):
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name='cookie_id'))
    result = add_cursor_features(ev, result)
    result = add_contact_features(ev, result)
    result = result.reset_index()[['cookie_id', *CURSOR_COLUMNS, *FLOW_COLUMNS]]
    return result

Координаты курсора беру только для платформ `web` и `desktop` и только когда есть обе координаты. Отсутствие курсора у мобильной куки само по себе ничего не доказывает. Здесь же считаю действия с контактами продавца и повторные обращения к одному объявлению.

## 5. Что смотрели другие куки

Это оказалось самым полезным добавлением. Сначала считаю совпадения по объявлениям в те же сутки. Потом смотрю, встречалось ли объявление в предыдущие 1–7 дней и насколько набор объявлений куки похож на наборы исторических соседей. Для текущего дня в этих последних признаках доступны только уже завершённые предыдущие дни. Метки других куки нигде не использую.

Популярное объявление могут смотреть и обычные люди. Одного совпадения недостаточно, поэтому модель учитывает его вместе с активностью, временем и другими признаками.

In [14]:
SHARED_ITEM_COLUMNS = [
    'peer_shared_item_count', 'peer_shared_item_fraction',
    'peer_cookie_count', 'peer_cookie_ge2_count', 'peer_max_item_overlap',
    'peer_item_mean_other_cookies', 'peer_item_max_other_cookies',
]


def same_day_shared_item_features(events, metadata):
    """Общие просмотры с другими куки за те же завершённые сутки, без меток."""
    start = pd.to_datetime(metadata.window_start_ts)
    end = pd.to_datetime(metadata.window_end_ts)
    assert (end - start).eq(pd.Timedelta(days=1)).all()
    assert start.dt.hour.eq(0).all() and start.dt.minute.eq(0).all()
    views = events.loc[events.event_name.eq('item_view') & events.item_id.notna(),
                       ['cookie_id', 'window_start_ts', 'item_id']].drop_duplicates().copy()
    if not views.empty:
        daily_count = views.groupby(['window_start_ts', 'item_id']).cookie_id.nunique()
        views['other_cookies'] = (
            views.set_index(['window_start_ts', 'item_id']).index.map(daily_count).to_numpy() - 1)
    else:
        views['other_cookies'] = pd.Series(dtype='int64')
    grouped = views.groupby('cookie_id')
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name='cookie_id'))
    n_items = grouped.item_id.size()
    shared = grouped.other_cookies.apply(lambda x: x.gt(0).sum())
    result['peer_shared_item_count'] = shared.reindex(result.index, fill_value=0)
    result['peer_shared_item_fraction'] = shared.div(n_items).reindex(result.index)
    result['peer_item_mean_other_cookies'] = grouped.other_cookies.mean().reindex(result.index)
    result['peer_item_max_other_cookies'] = grouped.other_cookies.max().reindex(result.index)

    # Считаю пересечения объявлений для пар куки одного дня.
    common = views.loc[views.other_cookies.gt(0),
                       ['cookie_id', 'window_start_ts', 'item_id']]
    # Сопоставляю куки, которые смотрели одно объявление в один день.
    pairs = common.merge(common, on=['window_start_ts', 'item_id'],
                         suffixes=('', '_peer'))
    pairs = pairs.loc[pairs.cookie_id.ne(pairs.cookie_id_peer)]
    overlap = pairs.groupby(['cookie_id', 'cookie_id_peer']).size()
    by_cookie = overlap.groupby(level='cookie_id')
    result['peer_cookie_count'] = by_cookie.size().reindex(result.index, fill_value=0)
    result['peer_cookie_ge2_count'] = overlap.ge(2).groupby(level='cookie_id').sum().reindex(
        result.index, fill_value=0)
    result['peer_max_item_overlap'] = by_cookie.max().reindex(result.index, fill_value=0)
    result = result.reset_index()[['cookie_id', *SHARED_ITEM_COLUMNS]]
    return result

Здесь совпадения считаются только среди куки одного дня. Повторные просмотры одной куки предварительно схлопываю. Мне нужно число других куки, а не число кликов по карточке.

In [15]:
HISTORY_COLUMNS = [
    'hist_prior7_item_count', 'hist_prior7_item_fraction',
    'hist_prior7_peer_mean', 'hist_prior7_peer_max',
    'hist_prior7_2plus_fraction', 'hist_prior7_nearest_day_median',
]


def seven_day_item_history(events, metadata):
    """Просмотры объявлений только за предыдущие семь суток."""
    views = events.loc[
        events.event_name.eq('item_view') & events.item_id.notna(),
        ['cookie_id', 'window_start_ts', 'item_id'],
    ].drop_duplicates()
    daily = views.groupby(['item_id', 'window_start_ts'], as_index=False).agg(
        day_cookies=('cookie_id', 'nunique'))
    # Для каждого объявления ищу дни, когда его уже смотрели.
    pairs = daily.merge(daily, on='item_id', suffixes=('', '_prior'))
    pairs['lag_days'] = (
        pairs.window_start_ts - pairs.window_start_ts_prior).dt.days
    pairs = pairs.loc[pairs.lag_days.between(1, 7)]
    history = pairs.groupby(['item_id', 'window_start_ts'], as_index=False).agg(
        prior7_peer_count=('day_cookies_prior', 'sum'),
        prior7_nearest_day=('lag_days', 'min'))
    enriched = views.merge(history, on=['item_id', 'window_start_ts'],
                           how='left', validate='many_to_one')
    enriched['prior7_peer_count'] = enriched.prior7_peer_count.fillna(0)
    enriched['has_prior'] = enriched.prior7_peer_count.gt(0)
    enriched['has_2plus_prior'] = enriched.prior7_peer_count.ge(2)
    by_cookie = enriched.groupby('cookie_id')
    out = pd.DataFrame(index=pd.Index(metadata.cookie_id, name='cookie_id'))
    out['hist_prior7_item_count'] = by_cookie.has_prior.sum().reindex(
        out.index, fill_value=0)
    out['hist_prior7_item_fraction'] = by_cookie.has_prior.mean().reindex(
        out.index)
    out['hist_prior7_peer_mean'] = by_cookie.prior7_peer_count.mean().reindex(
        out.index)
    out['hist_prior7_peer_max'] = by_cookie.prior7_peer_count.max().reindex(
        out.index)
    out['hist_prior7_2plus_fraction'] = by_cookie.has_2plus_prior.mean().reindex(
        out.index)
    out['hist_prior7_nearest_day_median'] = by_cookie.prior7_nearest_day.median().reindex(
        out.index)
    result = out.reset_index()[['cookie_id', *HISTORY_COLUMNS]]
    return result

Следующие признаки показывают, сколько просмотренных карточек встречалось у других куки за предыдущие 1–7 дней. Текущий и будущие дни в историю не входят.

In [16]:
ROUTE_COLUMNS = [
    'prior_route_peer_count',
    'prior_route_peer_ge2_count',
    'prior_route_peer_ge3_count',
    'prior_route_max_overlap',
    'prior_route_max_overlap_fraction',
    'prior_route_max_jaccard_ge2',
]


def prior_route_features(events, metadata):
    """Сходство с куки предыдущих 1–7 дней, без текущего и будущих дней."""
    views = events.loc[
        events.event_name.eq('item_view') & events.item_id.notna(),
        ['cookie_id', 'window_start_ts', 'item_id'],
    ].drop_duplicates()
    item_counts = views.groupby('cookie_id').item_id.nunique()
    # Соединяю просмотры по объявлению, затем оставляю только прошлые дни.
    pairs = views.merge(views, on='item_id', suffixes=('', '_prior'))
    lag = (pairs.window_start_ts - pairs.window_start_ts_prior).dt.days
    pairs = pairs.loc[lag.between(1, 7) &
                      pairs.cookie_id.ne(pairs.cookie_id_prior)]
    overlap = pairs.groupby(['cookie_id','cookie_id_prior']).item_id.nunique()
    overlap = overlap.rename('common_items').reset_index()
    overlap['self_items'] = overlap.cookie_id.map(item_counts)
    overlap['prior_items'] = overlap.cookie_id_prior.map(item_counts)
    overlap['jaccard'] = overlap.common_items / (
        overlap.self_items+overlap.prior_items-overlap.common_items)
    grouped = overlap.groupby('cookie_id')
    out = pd.DataFrame(index=pd.Index(metadata.cookie_id,name='cookie_id'))
    out['prior_route_peer_count'] = grouped.size().reindex(out.index,fill_value=0)
    out['prior_route_peer_ge2_count'] = grouped.common_items.apply(
        lambda x:x.ge(2).sum()).reindex(out.index,fill_value=0)
    out['prior_route_peer_ge3_count'] = grouped.common_items.apply(
        lambda x:x.ge(3).sum()).reindex(out.index,fill_value=0)
    out['prior_route_max_overlap'] = grouped.common_items.max().reindex(
        out.index,fill_value=0)
    out['prior_route_max_overlap_fraction'] = (
        out.prior_route_max_overlap / item_counts.reindex(out.index))
    out['prior_route_max_jaccard_ge2'] = overlap.loc[
        overlap.common_items.ge(2)].groupby('cookie_id').jaccard.max().reindex(out.index)
    result = out.reset_index()[['cookie_id',*ROUTE_COLUMNS]]
    return result

Последняя группа сравнивает уже целые наборы объявлений. Считаю, сколько исторических куки смотрели похожий набор и насколько велико пересечение. Если у двух куки совпали сразу несколько карточек, это уже больше похоже на общий маршрут.

## 6. Собираю одну таблицу на куку

Собираю все группы в одну таблицу. Здесь важно не перепутать события с повторами и без них. Порядок колонок фиксирую явно, чтобы валидация и финальное обучение использовали одинаковые признаки.

In [17]:
def own_cookie_features(metadata, with_duplicates, without_duplicates):
    parts = [
        basic_features(with_duplicates, metadata),
        temporal_features(with_duplicates, metadata),
        action_features(with_duplicates, metadata),
        content_features(without_duplicates, metadata),
        client_features(without_duplicates, metadata),
        sequence_features(without_duplicates, metadata),
        richer_behaviour_features(without_duplicates, metadata),
        same_day_shared_item_features(without_duplicates, metadata),
    ]
    table = parts[0]
    for part in parts[1:]:
        table = table.merge(part, on='cookie_id', validate='one_to_one')
    assert table.cookie_id.equals(metadata.cookie_id)
    return table


train_matrix = own_cookie_features(train, train_events, train_unique)
test_matrix = own_cookie_features(test, test_events, test_unique)
all_metadata = pd.concat([train, test], ignore_index=True)
all_events = pd.concat([train_unique, test_unique], ignore_index=True)
for feature_fn in (seven_day_item_history, prior_route_features):
    features = feature_fn(all_events, all_metadata)
    train_matrix = train_matrix.merge(features.iloc[:len(train)].reset_index(drop=True),
                                      on='cookie_id', validate='one_to_one')
    test_matrix = test_matrix.merge(features.iloc[len(train):].reset_index(drop=True),
                                    on='cookie_id', validate='one_to_one')

BASE_COLS = ['n_events', 'item_nunique']
COLS_120 = [*BASE_COLS, *TIME_COLUMNS, *ACTION_COUNT_COLUMNS, *CONTENT_COLUMNS,
            *CLIENT_COLUMNS, *SEQUENCE_COLUMNS, *CURSOR_COLUMNS, *FLOW_COLUMNS,
            *SHARED_ITEM_COLUMNS]
COLS_126 = [*COLS_120, *HISTORY_COLUMNS]
FINAL_COLS = [*COLS_126, *ROUTE_COLUMNS]
assert len(COLS_120) == 120 and len(COLS_126) == 126 and len(FINAL_COLS) == 132
assert len(FINAL_COLS) == len(set(FINAL_COLS))
assert train_matrix.cookie_id.equals(train.cookie_id)
assert test_matrix.cookie_id.equals(test.cookie_id)
assert set(FINAL_COLS).issubset(train_matrix.columns) and set(FINAL_COLS).issubset(test_matrix.columns)
assert not np.isinf(train_matrix[FINAL_COLS].to_numpy(dtype=float)).any()
assert not np.isinf(test_matrix[FINAL_COLS].to_numpy(dtype=float)).any()
print('Матрицы:', train_matrix.shape, test_matrix.shape)

Матрицы: (11091, 133) (4909, 133)


Отдельно проверяю, не попало ли будущее в признаки. После добавления test признаки train должны остаться прежними. Для каждого тестового дня удаляю все ещё более поздние дни и сравниваю признаки. Это не проверка качества, а проверка корректности расчёта.

In [18]:
for fn, columns in [(seven_day_item_history, HISTORY_COLUMNS),
                    (prior_route_features, ROUTE_COLUMNS)]:
    both = fn(all_events, all_metadata)
    only_train = fn(train_unique, train)
    pd.testing.assert_frame_equal(both.iloc[:len(train)].reset_index(drop=True), only_train)
    for day in sorted(test.window_start_ts.unique()):
        visible_meta = all_metadata.loc[all_metadata.window_start_ts.le(day)]
        visible_events = all_events.loc[all_events.window_start_ts.le(day)]
        today_ids = test.loc[test.window_start_ts.eq(day), 'cookie_id']
        visible = fn(visible_events, visible_meta).set_index('cookie_id').loc[today_ids]
        full = both.set_index('cookie_id').loc[today_ids]
        pd.testing.assert_frame_equal(visible, full)
print('Исторические признаки не используют будущие дни')

Исторические признаки не используют будущие дни


## 7. Валидация по времени

Train охватывает 6–19 апреля. Test идёт с 20 по 26 апреля. Для проверки всегда обучаюсь на более ранних днях и предсказываю более поздние. Сравниваю исходные 120 признаков, добавление истории объявлений (126) и маршрутов (132) при одинаковых настройках моделей. Семидневный период 13–19 апреля по длине совпадает с будущим тестовым периодом. Эти интервалы уже использовались при поиске решения, поэтому локальные числа не считаю гарантией скрытого результата.

In [19]:
CB_PARAMS = dict(iterations=800, depth=6, learning_rate=.05, l2_leaf_reg=5,
                 loss_function='Logloss', random_seed=SEED, thread_count=2,
                 allow_writing_files=False, verbose=False, nan_mode='Min')
HGB_PARAMS = dict(loss='log_loss', learning_rate=.05, max_iter=300,
                  max_leaf_nodes=7, min_samples_leaf=20, l2_regularization=1.0,
                  early_stopping=False, categorical_features=None, max_features=1.0,
                  class_weight=None, random_state=SEED)

def fit_ensemble(columns, fit_mask, valid_mask):
    cb = CatBoostClassifier(**CB_PARAMS)
    hgb = make_pipeline(
        SimpleImputer(strategy='constant', fill_value=-1, keep_empty_features=True),
        HistGradientBoostingClassifier(**HGB_PARAMS))
    with threadpool_limits(limits=2):
        cb.fit(train_matrix.loc[fit_mask, columns], train.loc[fit_mask, 'target'])
        hgb.fit(train_matrix.loc[fit_mask, columns], train.loc[fit_mask, 'target'])
        cb_score = cb.predict_proba(train_matrix.loc[valid_mask, columns])[:, 1]
        hgb_score = hgb.predict_proba(train_matrix.loc[valid_mask, columns])[:, 1]
    return .6 * cb_score + .4 * hgb_score

splits = [
    ('11–13 апреля', '2026-04-11', '2026-04-14'),
    ('14–16 апреля', '2026-04-14', '2026-04-17'),
    ('17–19 апреля', '2026-04-17', '2026-04-20'),
    ('13–19 апреля, неделя', '2026-04-13', '2026-04-20'),
]
validation_rows = []
for name, start, end in splits:
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    fit = train.window_end_ts.le(start)
    valid = train.window_start_ts.ge(start) & train.window_start_ts.lt(end)
    assert train.loc[fit, 'window_end_ts'].max() <= train.loc[valid, 'window_start_ts'].min()
    truth = train.loc[valid, 'target']
    for variant, columns in [('120 признаков', COLS_120),
                             ('126 признаков', COLS_126),
                             ('132 признака', FINAL_COLS)]:
        score = fit_ensemble(columns, fit, valid)
        validation_rows.append({
            'период': name, 'вариант': variant,
            'P@R≥0.70': precision_at_recall(truth, score),
            'Average Precision': average_precision_score(truth, score),
        })
    print('Готово:', name)
validation = pd.DataFrame(validation_rows)
display(validation.pivot(index='период', columns='вариант',
                         values='P@R≥0.70').round(4))

Готово: 11–13 апреля
Готово: 14–16 апреля
Готово: 17–19 апреля
Готово: 13–19 апреля, неделя


вариант,120 признаков,126 признаков,132 признака
период,,,
11–13 апреля,0.6393,0.6965,0.7268
"13–19 апреля, неделя",0.6456,0.6990,0.7259
14–16 апреля,0.6017,0.6493,0.7500
17–19 апреля,0.6957,0.7273,0.7619


Результаты локальной проверки ниже. Значения округлены, а ячейка выше считает их заново.

| Период проверки | 120 признаков | + история карточек, 126 | + похожие маршруты, 132 |
|---|---:|---:|---:|
| 11–13 апреля | 0,6393 | 0,6965 | 0,7268 |
| 14–16 апреля | 0,6017 | 0,6493 | 0,7500 |
| 17–19 апреля | 0,6957 | 0,7273 | 0,7619 |
| 13–19 апреля | 0,6456 | 0,6990 | 0,7259 |

На всех этих периодах новая группа помогла, хотя абсолютные значения заметно меняются от дат. Поэтому я не стал считать один удачный день доказательством будущего результата.

Результаты на коротких интервалах заметно различаются. Ботов в train меньше тысячи, поэтому небольшие срезы получаются шумными. Но в целом признаки предыдущих дней дали новый сигнал по сравнению с собственной историей куки. Я также пробовал поправку на популярность объявлений, более короткие интервалы истории, порядок просмотров, отдельные модели для платформ и несколько других алгоритмов. Некоторые варианты выигрывали на одном отрезке и проигрывали на другом, поэтому в итоговую отправку их не включил.

Веса 60/40 я оставил после более раннего сравнения с 50/50 на временных периодах. Разница была небольшой, но 60/40 чуть лучше на раннем и среднем периодах и не хуже на позднем. После добавления новых признаков вес заново не подбирал, чтобы не подстроиться под те же периоды.

## 8. Обучение на всём train и файл отправки

Теперь обучаю ту же смесь на всех размеченных куки. Порог я не подбираю. Платформа сама перебирает его по `score`. Обязательно соединяю оценки с образцом по `cookie_id`, потому что порядок строк в `sample_submission.csv` отличается от порядка в `test.csv`.

In [20]:
final_cb = CatBoostClassifier(**CB_PARAMS)
final_hgb = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=-1, keep_empty_features=True),
    HistGradientBoostingClassifier(**HGB_PARAMS))
with threadpool_limits(limits=2):
    final_cb.fit(train_matrix[FINAL_COLS], train.target)
    final_hgb.fit(train_matrix[FINAL_COLS], train.target)
    score = (.6 * final_cb.predict_proba(test_matrix[FINAL_COLS])[:, 1]
             + .4 * final_hgb.predict_proba(test_matrix[FINAL_COLS])[:, 1])

submission = sample[['cookie_id']].merge(
    test_matrix[['cookie_id']].assign(score=score),
    on='cookie_id', validate='one_to_one')
assert submission.columns.tolist() == ['cookie_id', 'score']
assert len(submission) == len(test) == len(sample)
assert submission.cookie_id.equals(sample.cookie_id)
assert submission.cookie_id.is_unique
assert np.isfinite(submission.score).all() and submission.score.between(0, 1).all()
submission.to_csv('submission.csv', index=False, float_format='%.17g')

print('Готово: submission.csv, строк:', len(submission))

Готово: submission.csv, строк: 4909


На платформе получилось 0,81360. Константный ответ дал 0,09391. Положительный класс здесь составлен из известных сервисов сбора данных, поэтому для новых сервисов качество может отличаться. Локальные временные проверки помогли выбрать вариант, но скрытый результат заранее не доказывали.

Для признаков использовал pandas и NumPy, для моделей scikit-learn и CatBoost. Все библиотеки работают локально. Внешних API и больших языковых моделей здесь нет. Ответы для отдельных `cookie_id` нигде не прописаны вручную.